# ⚽ Phase 3c — Validate football_v2c (ordinal + draw_signal) on the SAME real games

**This is a parallel run.** The original `phase3_wc26_validation.ipynb` graded
`football_v2.pth` against these games and that result is already permanently
logged as entry #1 in `results/RESULTS.md`. This notebook grades `football_v2c.pth`
(ordinal head + draw_signal feature) against the exact same 68 matches, so the two
numbers are finally a fair, apples-to-apples comparison — unlike the mistake in the
Phase 2c notebook where a WC-games number got compared against a club-football number.


Everything up to now was training and testing on data the model was allowed to learn
patterns from. This notebook is different: we grade `football_v2.pth` against the
**real, just-finished 2026 World Cup group stage** — games it has never seen a single
snapshot of.

The analogy: imagine a chef trained for months on thousands of ordinary home dinners
(the leagues). Tonight we don't hand them more practice — we hand them the **actual
order tickets from a real restaurant night that already happened**, and see how the
food came out. No do-overs, no peeking at the answer first. This is the real test.

## Data source
[worldcup26.ir](https://worldcup26.ir) — a free, open feed for the 2026 tournament.
Two endpoints:
- `/get/games` — every match, with scorer names **and goal minutes** baked into strings
  like `"R. Jiménez 67'"`
- `/get/teams` — team names, FIFA codes, groups

## Known simplifications (on the record, not hidden)
1. **Host-nation home advantage.** The 2026 World Cup is co-hosted by USA/Mexico/Canada.
   When USA plays *in* the USA, that's genuinely not a neutral venue for them — but our
   feature set marks every WC game `is_neutral_venue = 1`. We don't have a
   stadium-to-country mapping yet, so this is left as a known gap.
2. **Some 2026 debut teams have no real FIFA rank in our lookup table** (e.g. Curaçao,
   Cape Verde, Uzbekistan — new entries from the expanded 48-team format). They default
   to a mid-table rank of 50. This will make the model's "strength gap" reasoning less
   accurate specifically for those teams' matches.

This is exactly the kind of thing that belongs in a permanent results log rather than
memory — see the last cell.


## Cell 1 — Setup: load the trained v2c model (ordinal head)

In [2]:
import json, time, pickle, re, warnings
import numpy as np, pandas as pd
import requests
from pathlib import Path
from datetime import datetime, timezone
warnings.filterwarnings('ignore')

import torch, torch.nn as nn
from sklearn.metrics import accuracy_score, log_loss, confusion_matrix, classification_report

ROOT      = Path('.')
DATA_RAW  = ROOT / 'data' / 'raw'
MODELS    = ROOT / 'models'
RESULTS   = ROOT / 'results'
for d in (DATA_RAW, RESULTS): d.mkdir(parents=True, exist_ok=True)

# 12 features now -- draw_signal added at the end, same order as Phase 2c training
FEATURE_COLS = ['goal_diff','minute_norm','is_second_half','home_rank_norm',
                'away_rank_norm','rank_diff','is_knockout','lead_changes_norm',
                'is_neutral_venue','score_state','strength_x_time','draw_signal']

class FootballNetOrdinal(nn.Module):
    """Same architecture as Phase 2c: ordinal head instead of softmax."""
    def __init__(self, n_features=12, h1=40, h2=20, dropout=0.30):
        super().__init__()
        self.fc1 = nn.Linear(n_features, h1)
        self.fc2 = nn.Linear(h1, h2)
        self.s1_head = nn.Linear(h2, 1)
        self.f_head  = nn.Linear(h2, 1)
        self.drop = nn.Dropout(dropout); self.act = nn.ReLU()

    def raw_logits(self, x):
        """Return the two pre-sigmoid numbers -- needed so we can apply the
        saved temperature BEFORE sigmoid, exactly as Phase 2c calibrated it."""
        h = self.drop(self.act(self.fc1(x)))
        h = self.drop(self.act(self.fc2(h)))
        return self.s1_head(h).squeeze(-1), self.f_head(h).squeeze(-1)

    def forward(self, x):
        s1_logit, f_logit = self.raw_logits(x)
        s1 = torch.sigmoid(s1_logit); f = torch.sigmoid(f_logit)
        s2 = s1 * f
        return torch.stack([1 - s1, s1 - s2, s2], dim=1)

ckpt = torch.load(MODELS / 'football_v2c.pth', map_location='cpu', weights_only=False)
model = FootballNetOrdinal(**ckpt['arch'])
model.load_state_dict(ckpt['model_state'])
model.eval()
T_best = ckpt['temperature']
with open(MODELS / 'scaler_v2c.pkl', 'rb') as f:
    scaler = pickle.load(f)

print('Loaded football_v2c.pth')
print('Architecture:', ckpt['arch'])
print('Temperature T:', round(T_best, 3))
print('Warm-started from NBA:', ckpt['warm_started'])
print('Ordinal head:', ckpt.get('ordinal_head', False))


Loaded football_v2c.pth
Architecture: {'n_features': 12, 'h1': 40, 'h2': 20}
Temperature T: 0.881
Warm-started from NBA: False
Ordinal head: True


## Cell 2 — Fetch the 2026 World Cup data

We pull `games` and `teams` from the live feed and cache them locally, same pattern as
Phase 1's StatsBomb pull — first run downloads, every run after that is instant.


In [3]:
WC26_BASE = 'https://worldcup26.ir'

def fetch_wc26(endpoint):
    cache = DATA_RAW / f'wc26_{endpoint}.json'
    if cache.exists():
        return json.loads(cache.read_text())
    r = requests.get(f'{WC26_BASE}/get/{endpoint}', timeout=20)
    r.raise_for_status()
    data = r.json()
    cache.write_text(json.dumps(data))
    return data

games_data = fetch_wc26('games')
teams_data = fetch_wc26('teams')

all_games = games_data.get('games', games_data if isinstance(games_data, list) else [])
all_teams = teams_data.get('teams', teams_data if isinstance(teams_data, list) else [])

print(f'Total games in feed : {len(all_games)}')
print(f'Total teams in feed : {len(all_teams)}')

group_finished = [g for g in all_games if g.get('type')=='group' and str(g.get('finished')).upper()=='TRUE']
print(f'Finished GROUP-STAGE games: {len(group_finished)}  <- this is what we validate on')


Total games in feed : 104
Total teams in feed : 48
Finished GROUP-STAGE games: 72  <- this is what we validate on


## Cell 3 — Team strength lookup (FIFA codes) + the goal-minute parser

Two small pieces here.

**Team lookup:** the `teams` endpoint gives us `id -> fifa_code` (e.g. team id "9" -> "BRA").
We combine that with the same strength table from Phase 1 to get each team's 0-to-1
strength score.

**The scorer-string parser:** this is the regex we already proved works, pulled out into
a proper function. It finds every `<number>'` pattern (with optional `+<number>'` for
stoppage time) and ignores everything else in the string — names, own-goal tags, penalty
tags, and whether the quotes around it are straight or curly.


In [4]:
# Same curated strength table from Phase 1 (extended is fine to add to later)
FIFA_RANK = {
    'BRA':1,'ARG':2,'FRA':3,'ENG':4,'BEL':5,'NED':6,'POR':7,'ESP':8,'ITA':9,'GER':10,
    'CRO':11,'URU':12,'COL':13,'MEX':14,'USA':15,'SUI':16,'DEN':17,'SEN':18,'WAL':19,'IRN':20,
    'SRB':21,'MAR':22,'PER':23,'JPN':24,'SWE':25,'POL':26,'CHI':27,'KOR':28,'TUN':29,'CRC':30,
    'AUS':31,'NGA':32,'EGY':33,'SCO':34,'NOR':35,'TUR':36,'GHA':37,'ECU':38,'CIV':39,'QAT':40,
    'CAN':41,'CMR':42,'KSA':43,'IRQ':44,'RSA':45,'DZA':46,'CPV':47,'JOR':48,'UZB':49,'NZL':50,
    'CZE':51,'AUT':52,'BIH':53,'COD':54,'PAN':55,'SVK':56,
}
DEFAULT_RANK, MAX_RANK = 50, 100
def rank_to_norm(rank): return max(0.0, (MAX_RANK - rank) / (MAX_RANK - 1))
def get_strength(code): return rank_to_norm(FIFA_RANK.get(code, DEFAULT_RANK))

team_id_to_code = {t['id']: t.get('fifa_code', 'UNK') for t in all_teams}
unranked = sorted({team_id_to_code[t['id']] for t in all_teams
                   if team_id_to_code.get(t['id']) not in FIFA_RANK})
print(f'{len(team_id_to_code)} teams mapped to FIFA codes.')
print(f'{len(unranked)} teams default to rank 50 (no entry in our table):', unranked)

# ---- Goal-minute parser, proven against real examples in the previous cell ----
# FIXED: the feed uses TWO different stoppage-time formats, and the original
# pattern only caught one of them:
#   format A: "45'+5'"   -- apostrophe after BOTH the base minute and the extra
#   format B: "90+3'"    -- apostrophe ONLY after the extra, none after the base
# The old pattern required an apostrophe right after the base number, so format B
# silently lost the base minute entirely (a 93rd-minute goal was misread as minute 3).
_GOAL_RE = re.compile(r"(\d+)(?:'(?:\+(\d+)')?|\+(\d+)')")

def parse_scorers(raw):
    '''Turn a raw scorer string into a sorted list of goal minutes (capped at 90).'''
    if raw is None or str(raw).strip().lower() == 'null':
        return []
    minutes = []
    for base, extra_a, extra_b in _GOAL_RE.findall(str(raw)):
        extra = extra_a or extra_b or 0
        m = int(base) + int(extra)
        minutes.append(min(m, 90))
    return sorted(minutes)

# Self-test on the literal strings from the sample data — no guessing, proven correct
_t1 = parse_scorers('{"D. Bobadilla 7\'(OG)","F. Balogun 31\'","F. Balogun 45\'+5\'","G. Reyna 90\'+8\'"}')
_t2 = parse_scorers('null')
_t3 = parse_scorers('{"Armin Mhmich 90+3\'"}')   # the format-B case that broke before the fix
print('\nSelf-test: 4-goal string ->', _t1, ' (expect [7, 31, 50, 90])')
print('Self-test: null string    ->', _t2, ' (expect [])')
print('Self-test: 90+3\' format  ->', _t3, ' (expect [90]  <- capped from 93; old code gave [3])')


48 teams mapped to FIFA codes.
4 teams default to rank 50 (no entry in our table): ['ALG', 'CUW', 'HAI', 'PAR']

Self-test: 4-goal string -> [7, 31, 50, 90]  (expect [7, 31, 50, 90])
Self-test: null string    -> []  (expect [])
Self-test: 90+3' format  -> [90]  (expect [90]  <- capped from 93; old code gave [3])


## Cell 4 — Build snapshots for every finished group-stage match

Same idea as Phase 1's `build_snapshots`: freeze-frame the match every 5 minutes plus
right after every goal, and label every frame with the real final outcome.

We also run a **validation gate**: for every match, we count how many goal-minutes we
extracted for each side and compare it to the official `home_score` / `away_score`. If
they don't match, we print a warning and skip that match rather than silently feeding
the model bad data.


In [5]:
def build_snapshots(home_code, away_code, goal_events, is_knockout, is_neutral,
                    match_id, final_home, final_away):
    if   final_home > final_away: outcome = 2
    elif final_home < final_away: outcome = 0
    else:                         outcome = 1
    h_str, a_str = get_strength(home_code), get_strength(away_code)
    rank_diff = h_str - a_str
    goal_minutes = [m for (m,_) in goal_events]
    checkpoints = sorted(set([0] + list(range(5,91,5)) + goal_minutes + [90]))
    rows, lead_changes, prev_leader = [], 0, 0
    for minute in checkpoints:
        hs = sum(1 for m,s in goal_events if m<=minute and s=='home')
        as_= sum(1 for m,s in goal_events if m<=minute and s=='away')
        leader = (hs>as_) - (hs<as_)
        if leader != prev_leader and leader != 0: lead_changes += 1
        prev_leader = leader
        diff = int(np.clip(hs-as_, -5, 5))
        minute_norm = min(minute/90.0, 1.0)
        score_state = 0 if diff<0 else (2 if diff>0 else 1)
        rows.append({
            'match_id': match_id, 'minute': minute,
            'goal_diff': diff, 'minute_norm': round(minute_norm,4),
            'is_second_half': 1 if minute>45 else 0,
            'home_rank_norm': round(h_str,4), 'away_rank_norm': round(a_str,4),
            'rank_diff': round(rank_diff,4), 'is_knockout': int(is_knockout),
            'lead_changes_norm': round(lead_changes/max(1, hs+as_), 4),  # FIXED: goals-so-far, not final total
            'is_neutral_venue': int(is_neutral), 'score_state': score_state,
            'strength_x_time': round(rank_diff*(1.0-minute_norm),4),
            'outcome': outcome, 'home_code': home_code, 'away_code': away_code,
        })
    return rows

wc26_rows = []
mismatches = []
matchday_lookup = {}   # match_id -> matchday (for the streak analysis later)

for g in group_finished:
    hc = team_id_to_code.get(g['home_team_id'], 'UNK')
    ac = team_id_to_code.get(g['away_team_id'], 'UNK')
    fh, fa = int(g['home_score']), int(g['away_score'])

    home_goals = parse_scorers(g.get('home_scorers'))
    away_goals = parse_scorers(g.get('away_scorers'))

    if len(home_goals) != fh or len(away_goals) != fa:
        mismatches.append((g['id'], hc, ac, fh, fa, len(home_goals), len(away_goals)))
        continue

    events = sorted([(m,'home') for m in home_goals] + [(m,'away') for m in away_goals])
    match_id = f"wc26_{g['id']}"
    wc26_rows += build_snapshots(hc, ac, events, is_knockout=0, is_neutral=1,
                                 match_id=match_id, final_home=fh, final_away=fa)
    matchday_lookup[match_id] = g.get('matchday', '?')

wc26_df = pd.DataFrame(wc26_rows)

# Add draw_signal -- same Dixon-Coles-inspired formula as Phase 2c,
# derived from columns we already have, nothing new to fetch.
wc26_df['draw_signal'] = ((1 - wc26_df['rank_diff'].abs()).clip(0,1) *
                         (1 - wc26_df['goal_diff'].abs()/5).clip(0,1))
print(f'Snapshots built : {len(wc26_df):,}  from {wc26_df.match_id.nunique()} matches')
print(f'Matches skipped due to goal-count mismatch: {len(mismatches)}')
for m in mismatches[:10]:
    print('  mismatch:', m)


Snapshots built : 1,439  from 68 matches
Matches skipped due to goal-count mismatch: 4
  mismatch: ('20', 'AUT', 'JOR', 3, 1, 2, 1)
  mismatch: ('26', 'SUI', 'BIH', 4, 1, 3, 1)
  mismatch: ('28', 'CZE', 'RSA', 1, 1, 1, 0)
  mismatch: ('72', 'COD', 'UZB', 3, 1, 2, 1)


## Cell 4b — Why did 4 matches get skipped?

Four matches had a goal count that didn't match the official score: Austria-Jordan,
Switzerland-Bosnia, Czechia-South Africa, DR Congo-Uzbekistan. Rather than just
discarding them, let's look at the raw scorer strings directly — this is the same habit
as Phase 1's self-tests: don't just accept "it broke," go look at *why*.

The likely suspect: an **own goal that benefited the other team**, but got listed in
the *scoring* team's array rather than the *benefiting* team's array. Recall the
Bobadilla case from before — that one happened to already sit in the correct team's
list. If one of these four has the same tag sitting in the *wrong* list, our count
would be off by exactly one goal for each side, which is exactly the shape of small
mismatches (1-goal short on one side) we'd expect to see below.


**Update after running this:** all four turned out to be the same simple cause —
the feed is just missing a scorer entry for one goal in each match (or, for the
Czechia game, not crediting the goal to either side at all). Not an own-goal
mix-up, just a small gap in the free data source. Correctly skipping these 4
matches was the right call — nothing to fix here.


In [6]:
mismatch_ids = {m[0] for m in mismatches}  # ('20','26','28','72')

for g in all_games:
    if g['id'] in mismatch_ids:
        print(f"Match {g['id']}: {g['home_team_name_en']} {g['home_score']} - "
              f"{g['away_score']} {g['away_team_name_en']}")
        print(f"  home_scorers raw: {g.get('home_scorers')}")
        print(f"  away_scorers raw: {g.get('away_scorers')}")
        print(f"  parsed home goals: {parse_scorers(g.get('home_scorers'))}")
        print(f"  parsed away goals: {parse_scorers(g.get('away_scorers'))}")
        print()


Match 20: Austria 3 - 1 Jordan
  home_scorers raw: {"Rvmanv Ashmid 21'","Izn Alarb 76'"}
  away_scorers raw: {"Ali Avlvan 50'"}
  parsed home goals: [21, 76]
  parsed away goals: [50]

Match 26: Switzerland 4 - 1 Bosnia and Herzegovina
  home_scorers raw: {"Jvhan Mnzambi 74'","Rvbn Vargas 84'","Jvhan Mnzambi 90'"}
  away_scorers raw: {"Armin Mhmich 90+3'"}
  parsed home goals: [74, 84, 90]
  parsed away goals: [90]

Match 28: Czech Republic 1 - 1 South Africa
  home_scorers raw: {"‫mikhal Sadilk 6'"}
  away_scorers raw: null
  parsed home goals: [6]
  parsed away goals: []

Match 72: Democratic Republic of the Congo 3 - 1 Uzbekistan
  home_scorers raw: {"Fistvn Mail 78'","Yoane Wissa 90+1'"}
  away_scorers raw: {"Aldvr Shvmvrvdvf 10'"}
  parsed home goals: [78, 90]
  parsed away goals: [10]



## Cell 5 — Run the model on real 2026 games

We scale the features with the *same scaler fitted back in Phase 2* (never refit a
scaler on new data — that would be like changing your measuring cup between the recipe
and the cooking), then get the model's 3 probabilities per snapshot.


In [7]:
X26 = wc26_df[FEATURE_COLS].values.astype('float32')
y26 = wc26_df['outcome'].values.astype('int64')
X26_scaled = scaler.transform(X26).astype('float32')

model.eval()
with torch.no_grad():
    s1_logit, f_logit = model.raw_logits(torch.tensor(X26_scaled))
    s1_logit = s1_logit.numpy(); f_logit = f_logit.numpy()

def probs_at_T(s1_logit, f_logit, T):
    s1 = 1/(1+np.exp(-s1_logit/T)); f = 1/(1+np.exp(-f_logit/T))
    s2 = s1*f
    return np.stack([1-s1, s1-s2, s2], axis=1)

probs26 = probs_at_T(s1_logit, f_logit, T_best)
preds26 = probs26.argmax(1)
print(f'Ran inference on {len(X26_scaled):,} real 2026 World Cup snapshots.')

# Prove the ordinal guarantee held here too, on live tournament data
print(f'P(draw) never negative : {bool((probs26[:,1] >= -1e-6).all())}')
print(f'Always sums to 1       : {bool(np.allclose(probs26.sum(axis=1), 1.0, atol=1e-4))}')

Ran inference on 1,439 real 2026 World Cup snapshots.
P(draw) never negative : True
Always sums to 1       : True


## Cell 6 — The honest grade, and the club-football-vs-World-Cup comparison

Two accuracy numbers side by side answer the actual question this phase exists to ask:
**"can a model mostly trained on club football read international tournament
football?"**
- `v2 test accuracy` — Phase 2's grade, on games from the *same distribution* it trained on.
- `2026 WC accuracy` — this phase's grade, on games from a *different* distribution
  (international, neutral venue, one-off knockout pressure, national pride).

If the WC number is close to the test number, the transfer worked. If it's much worse,
that's honest evidence international football plays differently to club football —
which would itself be a useful, real finding.


In [8]:
names = ['away','draw','home']
overall_acc  = accuracy_score(y26, preds26)
baseline_acc = max(np.bincount(y26, minlength=3)) / len(y26)
ll = log_loss(y26, probs26, labels=[0,1,2])

print('='*58)
print('2026 WORLD CUP GROUP STAGE — REAL GAME VALIDATION')
print('='*58)
print(f'Snapshots graded      : {len(y26):,}  ({wc26_df.match_id.nunique()} matches)')
print(f'Accuracy (2026 WC)    : {overall_acc:.3f}')
print(f'Baseline (always home): {baseline_acc:.3f}')
print(f'Log-loss              : {ll:.4f}')
print()
print('For reference:')
print('  v2  (softmax)  on THIS SAME real WC data: accuracy 0.712, log-loss 0.6382')
print('  v2  (softmax)  on club-football test set : accuracy 0.550, log-loss 0.8782')
print('  v2c (ordinal)  on club-football test set : accuracy 0.551, log-loss 0.9030')
print()
print('Confusion matrix (rows = truth, cols = guess):')
cm = confusion_matrix(y26, preds26, labels=[0,1,2])
print('           guess_away  guess_draw  guess_home')
for i,n in enumerate(names):
    print(f'  true_{n}  ' + ''.join(f'{cm[i][j]:11d}' for j in range(3)))
print()
print(classification_report(y26, preds26, target_names=names, digits=3, zero_division=0))


2026 WORLD CUP GROUP STAGE — REAL GAME VALIDATION
Snapshots graded      : 1,439  (68 matches)
Accuracy (2026 WC)    : 0.661
Baseline (always home): 0.468
Log-loss              : 0.6931

For reference:
  v2  (softmax)  on THIS SAME real WC data: accuracy 0.712, log-loss 0.6382
  v2  (softmax)  on club-football test set : accuracy 0.550, log-loss 0.8782
  v2c (ordinal)  on club-football test set : accuracy 0.551, log-loss 0.9030

Confusion matrix (rows = truth, cols = guess):
           guess_away  guess_draw  guess_home
  true_away          352          0         29
  true_draw          111         63        211
  true_home          104         33        536

              precision    recall  f1-score   support

        away      0.621     0.924     0.743       381
        draw      0.656     0.164     0.262       385
        home      0.691     0.796     0.740       673

    accuracy                          0.661      1439
   macro avg      0.656     0.628     0.581      1439
weighte

## Cell 7 — The REAL track record: accuracy at kickoff and half-time

**A correction from the first run.** Grading the model using each match's *final*
snapshot (minute 90) was misleading — by minute 90 the score already tells you who won,
so a near-100% "accuracy" there just proves the model can read a scoreline, not that it
has real football skill. It's like judging someone's weather forecasting by asking
"is it raining right now?" while they're standing outside getting wet — technically a
correct answer, but it proves nothing about forecasting.

The genuine track record needs to grade the model at a point where the outcome is
**still genuinely uncertain**:

- **Kickoff** (minute 0): pure pre-game guess, based only on team strength. `goal_diff`
  is 0 for every match here — the model has nothing to go on except which team is rated
  stronger. This tests raw team-strength judgement.
- **Half-time** (minute ~45): score plus context, but the game is still very much open.
  This is the honest mid-match skill test.

Comparing kickoff accuracy vs half-time accuracy also answers a nice side question: how
much does actually watching the first 45 minutes help, on top of just knowing team
strength beforehand?


In [9]:
# FIXED: the previous version used groupby().apply() with a function that
# returns a single row. Pandas then rebuilds the result indexed by MATCH_ID
# (the group key) instead of the original row numbers. The next step tried to
# look up predictions using that mismatched index, silently failed for every
# single row, and numpy's negative-indexing quirk (index -1 = "last item")
# meant EVERY row quietly got the prediction for the very last snapshot in the
# whole dataset. That's why kickoff and half-time came out identical -- both
# were actually grading the same one wrong answer against everything.
#
# The fix: give every row an explicit, safe integer position up front, and use
# THAT to look up its prediction -- no ambiguous pandas index involved at all.

wc26_df = wc26_df.reset_index(drop=True)
wc26_df['row_pos'] = np.arange(len(wc26_df))   # row_pos[i] always matches preds26[i]
wc26_df['matchday'] = wc26_df['match_id'].map(matchday_lookup)

def nearest_snapshot(group, target_minute):
    '''Pick the snapshot in this match closest to (but not after) target_minute.'''
    sub = group[group['minute'] <= target_minute]
    if sub.empty:
        sub = group
    return sub.sort_values('minute').iloc[-1]

def build_grade_table(target_minute):
    rows = []
    for match_id, g in wc26_df.groupby('match_id'):
        r = nearest_snapshot(g, target_minute)
        pos  = int(r['row_pos'])          # the safe, explicit position
        pred = preds26[pos]               # guaranteed to be THIS row's real prediction
        rows.append({
            'match_id': match_id, 'matchday': r['matchday'], 'minute': int(r['minute']),
            'outcome': int(r['outcome']), 'pred': int(pred),
            'correct': int(pred == r['outcome']),
        })
    return pd.DataFrame(rows)

kickoff_graded  = build_grade_table(0)
halftime_graded = build_grade_table(45)

print('='*66)
print('GENUINE TRACK RECORD — accuracy when the outcome was still uncertain')
print('='*66)
print(f'{"Matchday":<10}{"Matches":>9}{"Kickoff acc":>13}{"Half-time acc":>15}')
for m_ in sorted(kickoff_graded['matchday'].unique()):
    ko = kickoff_graded[kickoff_graded['matchday']==m_]
    ht = halftime_graded[halftime_graded['matchday']==m_]
    print(f'{m_:<10}{len(ko):>9}{ko["correct"].mean():>13.3f}{ht["correct"].mean():>15.3f}')

print()
print(f'Overall kickoff accuracy  (pure pre-game strength guess): {kickoff_graded["correct"].mean():.3f}')
print(f'Overall half-time accuracy (score + context, still open) : {halftime_graded["correct"].mean():.3f}')
print(f'Overall full-time "accuracy" (reading a final score)     : {overall_acc:.3f}  <- NOT a skill measure, kept only for context')
print()
print('Reading this: kickoff accuracy is the honest floor (strength alone).')
print('Half-time should sit above it if watching the first 45 minutes genuinely helps.')
print('Full-time should be highest of all -- that\'s expected and fine, it is not the interesting number.')

GENUINE TRACK RECORD — accuracy when the outcome was still uncertain
Matchday    Matches  Kickoff acc  Half-time acc
1                23        0.478          0.522
2                22        0.636          0.682
3                23        0.565          0.609

Overall kickoff accuracy  (pure pre-game strength guess): 0.559
Overall half-time accuracy (score + context, still open) : 0.603
Overall full-time "accuracy" (reading a final score)     : 0.661  <- NOT a skill measure, kept only for context

Reading this: kickoff accuracy is the honest floor (strength alone).
Half-time should sit above it if watching the first 45 minutes genuinely helps.
Full-time should be highest of all -- that's expected and fine, it is not the interesting number.


### Spotlight, corrected: watch the probability move, not just the final answer

Rather than only showing the final-whistle call (which is close to trivial once the
score is 7-1), we now print the model's probability **at kickoff, at half-time, and at
full-time** for the same outlier game — so you can actually see it react as the goals
go in.


In [10]:
def prob_at(match_id, target_minute):
    sub = wc26_df[wc26_df.match_id == match_id]
    row = nearest_snapshot(sub, target_minute)
    pos = int(row['row_pos'])   # same safe lookup as the fixed track-record cell
    return row, probs26[pos]

print('Spotlight: GER vs CUW (final score 7-1) -- the probability journey')
for label, minute in [('kickoff', 0), ('half-time', 45), ('full-time', 90)]:
    row, p = prob_at('wc26_10', minute)
    print(f'  {label:<10} (min {int(row.minute):>2}, score {int(row.goal_diff):+d}): '
          f'away {p[0]:.1%}  draw {p[1]:.1%}  home {p[2]:.1%}')

Spotlight: GER vs CUW (final score 7-1) -- the probability journey
  kickoff    (min  0, score +0): away 13.0%  draw 33.2%  home 53.8%
  half-time  (min 45, score +1): away 4.6%  draw 19.2%  home 76.1%
  full-time  (min 90, score +5): away 0.0%  draw 0.0%  home 100.0%


## Cell 8 — Log this to the permanent track record

In [11]:
stamp = datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M UTC')
entry = f'''
## Phase 3 — REAL 2026 World Cup group-stage validation ({stamp})
**This is entry #1 of the real-games track record.**
- Model graded: football_v2c.pth (ordinal head + draw_signal feature)
- Matches graded: {wc26_df.match_id.nunique()}  (skipped {len(mismatches)} due to goal-count mismatch)
- Snapshots graded: {len(y26):,}
- Accuracy: {overall_acc:.3f}  (baseline always-home: {baseline_acc:.3f})
- Log-loss: {ll:.4f}
- For comparison: v2 (softmax) scored 0.712 accuracy / 0.6382 log-loss on these SAME real games
- Known caveats: host-nation home advantage not modelled; some 2026 debut teams
  default to rank 50 (no real FIFA rank in our table)
'''
rp = RESULTS / 'RESULTS.md'
if not rp.exists(): rp.write_text('# World Cup 2026 Win Probability — Results Log\n')
with rp.open('a') as f: f.write(entry)

mc = RESULTS / 'metrics.csv'
row = pd.DataFrame([{'timestamp':stamp,'phase':'phase3c_wc26c_validation',
    'test_acc':round(overall_acc,4),'baseline_acc':round(baseline_acc,4),
    'test_logloss':round(ll,4),'matches':wc26_df.match_id.nunique()}])
row.to_csv(mc, mode='a', header=not mc.exists(), index=False)

print(entry)
print('PHASE 3 COMPLETE — track record has begun.')



## Phase 3 — REAL 2026 World Cup group-stage validation (2026-07-04 10:23 UTC)
**This is entry #1 of the real-games track record.**
- Model graded: football_v2c.pth (ordinal head + draw_signal feature)
- Matches graded: 68  (skipped 4 due to goal-count mismatch)
- Snapshots graded: 1,439
- Accuracy: 0.661  (baseline always-home: 0.468)
- Log-loss: 0.6931
- For comparison: v2 (softmax) scored 0.712 accuracy / 0.6382 log-loss on these SAME real games
- Known caveats: host-nation home advantage not modelled; some 2026 debut teams
  default to rank 50 (no real FIFA rank in our table)

PHASE 3 COMPLETE — track record has begun.
